# 13 · Follow-up recommendation at the 12-month visit (P7)

Alert level (green / amber / red), next visit (12 / 6 / 3 months) and review prompts, from the predicted trajectory and the recalibrated risks. Thresholds are placeholders in `config.yaml` (followup.alerts) until agreed with clinicians. Code: `followup/recommend.py`.

> **Data source:** simulated post-transplant cohort calibrated to published studies (method development), because NEBSL follow-up records are not yet available. The same code runs on real follow-up data (`data_source = "nebsl"`).
>
> **Before committing, clear all outputs** (VS Code: `...` → *Clear All Outputs* → save).

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `recommendation` importable whether this runs from notebooks/, recommendation/ or ml/
ML = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "recommendation" / "config.py").exists())
sys.path.insert(0, str(ML))

import pandas as pd
from IPython.display import Image, Markdown, display
from recommendation.config import load_config

pd.set_option("display.width", 160)
cfg = load_config()
P = cfg["paths"]
OUTPUTS = Path(P["outputs"])
MODELS = Path(P["models"])
print("Cohort:", Path(P["followup_cohort"]).name, "| landmark dataset:", Path(P["followup_dataset"]).name)

In [ ]:
import joblib
from recommendation.data_prep.followup_dataset import load
from recommendation.followup import recommend as R

table, _ = load(P["followup_dataset"], MODELS / "followup_preprocessor.json")
traj, risk = joblib.load(MODELS / "followup_trajectory.joblib"), joblib.load(MODELS / "followup_risk.joblib")
rules = cfg["followup"]["alerts"]
test = table[table["split"] == "test"]
rec = R.recommend(test, traj, risk, rules)
rec["alert"].value_counts()

## 1. What happened next, per alert level (test grafts)

In [ ]:
out = R.outcomes_by_alert(rec, test)
display(out)
print("Missed deterioration:", R.safety(out), "| visits:", R.visit_burden(rec))

## 2. The same on a separate 3,000-graft simulated cohort (never used for fitting)

In [ ]:
r = json.loads((OUTPUTS / "followup" / "followup_recommendations.json").read_text())
e = r["evaluation"]["external_simulated_cohort"]
display(pd.DataFrame(e["by_alert"]))
print(e["safety"], e["visits"])

## 3. Three example grafts

In [ ]:
cols = ["graft_id", "alert", "next_visit_months", "ecd_12m", "ecd_24m", "ecd_60m_low", "failure_risk_24m",
        "low_ecd_risk_24m", "rules_fired", "actions"]
pd.concat([rec[rec["alert"] == a].head(1) for a in ("red", "amber", "green")])[cols].round(3)